# Phase 1 — Feasibility Checks

**Purpose:** Determine whether the dataset supports the intended hierarchical model.
All four checks must pass or have confirmed fallbacks before proceeding.

In [1]:
import pandas as pd

df1 = pd.read_excel("../data/online_retail_II.xlsx", sheet_name="Year 2009-2010")
df2 = pd.read_excel("../data/online_retail_II.xlsx", sheet_name="Year 2010-2011")
df = pd.concat([df1, df2], ignore_index=True)

print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
print(df.head())

(1067371, 8)
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

          InvoiceDate  Price  Customer ID         Country  
0 2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2 2009-12-01 07:45:00   6.75  

## CHECK 1 — Country segment distribution

**This is the most important feasibility check.**

- **Go (ideal):** UK dominates but at least 8–10 countries have 50–500 unique customers.
- **Go (acceptable):** Fewer than 8 countries in middle tier — restrict to top 15.
- **No-go:** Fewer than 5 countries with >30 customers outside UK. Fallback: switch to product division grouping.

In [2]:
seg = (
    df.groupby("Country")["Customer ID"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"Customer ID": "n_customers"})
)
print(seg.to_string())
print(f"\nTotal countries: {len(seg)}")
print(f"Countries with >50 customers: {(seg.n_customers > 50).sum()}")
print(f"Countries with 10-500 customers: {seg.n_customers.between(10,500).sum()}")

                 Country  n_customers
0         United Kingdom         5410
1                Germany          107
2                 France           95
3                  Spain           41
4                Belgium           29
5               Portugal           24
6            Netherlands           23
7            Switzerland           22
8                 Sweden           19
9                  Italy           17
10             Australia           15
11               Finland           15
12       Channel Islands           14
13                Norway           13
14               Austria           13
15               Denmark           12
16                Cyprus           11
17                 Japan           10
18                   USA            9
19           Unspecified            7
20                Poland            6
21                  EIRE            5
22                Greece            5
23                Canada            5
24  United Arab Emirates            4
25          

## CHECK 2 — Customer ID missingness

- **Go:** Overall missingness <35%. Drop rows with missing Customer ID.
- **Marginal (35–50%):** Proceed but flag in README. Check if missingness correlates with country/time.
- **No-go (>50%):** Switch to invoice-level repeat purchase rate.

**Research note:** Missing Customer ID likely reflects guest checkouts (MNAR). Do not impute.

In [3]:
overall_missing = df["Customer ID"].isna().mean()
print(f"Overall Customer ID missing: {overall_missing:.1%}")

missing_by_country = (
    df.groupby("Country")["Customer ID"]
    .apply(lambda x: x.isna().mean())
    .sort_values(ascending=False)
)
print(missing_by_country)

Overall Customer ID missing: 22.8%
Country
Bermuda                 1.000000
Hong Kong               1.000000
Bahrain                 0.531746
Unspecified             0.306878
RSA                     0.272189
United Kingdom          0.244596
United Arab Emirates    0.228000
Lebanon                 0.224138
Israel                  0.126685
EIRE                    0.093530
Nigeria                 0.062500
Portugal                0.044275
Switzerland             0.039197
Sweden                  0.013930
France                  0.008932
Denmark                 0.000000
Brazil                  0.000000
Belgium                 0.000000
Austria                 0.000000
Australia               0.000000
Czech Republic          0.000000
Cyprus                  0.000000
Channel Islands         0.000000
Canada                  0.000000
Germany                 0.000000
Finland                 0.000000
European Community      0.000000
Netherlands             0.000000
Malta                   0.000000


## CHECK 3 — Churn base rate and class balance

- **Acceptable range:** 20–70% churn rate.
- If outside this range, adjust the churn window (try 60 or 120 days) and re-check.

In [5]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
max_date = df["InvoiceDate"].max()
cutoff = max_date - pd.Timedelta(days=90)

customer_last = (
    df.dropna(subset=["Customer ID"])
    .groupby("Customer ID")["InvoiceDate"]
    .max()
    .reset_index()
    .rename(columns={"InvoiceDate": "last_purchase"})
)
customer_last["churned"] = (customer_last["last_purchase"] < cutoff).astype(int)

print(f"Churn base rate: {customer_last.churned.mean():.1%}")
print(f"Total identified customers: {len(customer_last)}")

Churn base rate: 50.9%
Total identified customers: 5942


## CHECK 4 — PyMC sampling speed

- **<120s:** Full model tractable with NUTS.
- **120–300s:** Tractable but slow. Use top 10 countries only.
- **>300s:** Switch to ADVI (variational inference).

In [ ]:
import pymc as pm
import numpy as np
import time

sample_df = customer_last.sample(5000, random_state=42)
outcome = sample_df["churned"].values
idx = (sample_df.index % 2).values

with pm.Model() as timing_model:
    mu = pm.Normal("mu", 0, 1)
    tau = pm.HalfNormal("tau", 1)
    beta = pm.Normal("beta", mu, tau, shape=2)
    p = pm.math.sigmoid(beta[idx])
    y = pm.Bernoulli("y", p=p, observed=outcome)
    
    t0 = time.time()
    trace = pm.sample(200, tune=200, cores=1, progressbar=True, random_seed=42)
    t1 = time.time()

print(f"Elapsed: {t1-t0:.0f}s for 200 samples + 200 tune, 5k obs, 2 segments")

WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.
/home/rueschenpoehler/bayesian-segmentation/.venv/lib/python3.11/site-packages/arviz/__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [mu, tau, beta]


Output()

## Phase 1 Exit Criteria Summary

Fill in after running all checks:

1. **Grouping variable:** Country / Product division
2. **Missingness rate:** __% — Handling: drop / flag / switch to invoice-level
3. **Churn window:** __ days — Base rate: __%
4. **Sampling strategy:** NUTS / ADVI